# Pedestrian Detection using HOG + SVM

**Histogram of Oriented Gradients (HOG)** captures the distribution of local gradient directions in a dense grid of cells, producing a compact, discriminative descriptor for object shapes.  
A **Support Vector Machine (SVM)** trained on HOG descriptors is the canonical approach to pedestrian (and general rigid-object) detection.

## Dataset
[**INRIA Person**](https://huggingface.co/datasets/marcelarosalesj/inria-person) — loaded directly from HuggingFace (~239 MB, cached after first download).  
The dataset contains pre-cropped image patches in two classes:

| Label | Folder | Meaning |
|-------|--------|---------|
| `1` | `data_ped/pedestrians/` | Cropped pedestrian patches (positive) |
| `0` | `data_ped/no_pedestrians/` | Background patches (negative) |

## Pipeline
| Step | Description |
|------|-------------|
| 1 | Load dataset via `datasets.load_dataset` — images as PIL objects |
| 2 | Resize every patch to **128 × 64** px; compute **HOG descriptor** (9 orientations, 8×8 px cells, 2×2 cell blocks) |
| 3 | Split into 80 % train / 20 % test (stratified) |
| 4 | Train an **SVM** classifier (RBF kernel, grid-searched C & γ) |
| 5 | Evaluate with accuracy, precision, recall, F1, and ROC-AUC |
| 6 | Visualise predicted labels on sample test patches

---
## 1 · Imports & Configuration

In [ ]:
# Install dependencies (safe to re-run — already present packages are skipped)
%pip install -q opencv-python-headless scikit-image scikit-learn datasets Pillow tqdm

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pickle

from PIL import Image
from datasets import load_dataset

from skimage.feature import hog
from skimage import exposure

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from tqdm import tqdm

print(f"OpenCV    : {cv2.__version__}")
print(f"NumPy     : {np.__version__}")

# ── Global configuration ───────────────────────────────────────────────────
CONFIG = {
    'dataset_id'  : 'marcelarosalesj/inria-person',
    'test_size'   : 0.20,        # 80 % train / 20 % test
    'random_seed' : 42,
    # Detection window  (H × W) — standard HOG pedestrian window
    'window_size' : (128, 64),
    # HOG parameters (Dalal & Triggs 2005 defaults)
    'hog': {
        'orientations'   : 9,
        'pixels_per_cell': (8, 8),
        'cells_per_block': (2, 2),
        'block_norm'     : 'L2-Hys',
        'transform_sqrt' : True,
    },
    'model_path'  : 'hog_svm_model.pkl',
    'scaler_path' : 'hog_svm_scaler.pkl',
}

print("\nConfiguration:")
for k, v in CONFIG.items():
    print(f"  {k:<20} : {v}")

---
## 2 · HOG Feature Extraction

The HOG descriptor divides the detection window into small spatial **cells** (8×8 px).  
Within each cell a histogram of **9 gradient orientation bins** is computed.  
Neighbouring cells are grouped into **blocks** (2×2 cells) and L2-Hys normalised for illumination invariance.

In [ ]:
def compute_hog(image: np.ndarray, visualize: bool = False):
    """Resize patch to detection window, convert to gray, return HOG features."""
    H, W = CONFIG['window_size']
    resized = cv2.resize(image, (W, H))
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY) if resized.ndim == 3 else resized

    return hog(
        gray,
        visualize=visualize,
        **CONFIG['hog'],
    )


# ── Dimension check on a blank patch ──────────────────────────────────────
dummy = np.zeros((*CONFIG['window_size'], 3), dtype=np.uint8)
feat_len = len(compute_hog(dummy))
print(f"HOG descriptor length : {feat_len} dimensions")

---
## 3 · HOG Visualisation

We preview HOG on a real INRIA pedestrian patch to confirm the descriptor captures the dominant edge orientations (limbs, head, shoulders).  
A tiny `train[:3]` slice is streamed from HuggingFace so no full download is required at this stage.

In [ ]:
# Stream 3 samples just for visualization (fast — no full download yet)
ds_preview = load_dataset(CONFIG['dataset_id'], split='train[:3]')

# Find a pedestrian sample (label == 1) if available, else use first sample
preview_sample = next(
    (ex for ex in ds_preview if ex['label'] == 1),
    ds_preview[0]
)

# PIL → uint8 BGR numpy array
sample_rgb = np.array(preview_sample['image'].convert('RGB'))
sample_bgr = cv2.cvtColor(sample_rgb, cv2.COLOR_RGB2BGR)

H, W = CONFIG['window_size']
patch      = cv2.resize(sample_bgr, (W, H))
patch_gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)

features, hog_img = hog(
    patch_gray,
    **{k: v for k, v in CONFIG['hog'].items()},
    visualize=True,
)
hog_img_rescaled = exposure.rescale_intensity(hog_img, in_range=(0, 10))

label_str = 'pedestrian' if preview_sample['label'] == 1 else 'background'

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"INRIA patch — {label_str}\nresized to {H}×{W} px")
axes[0].axis('off')

axes[1].imshow(hog_img_rescaled, cmap='gray')
axes[1].set_title(f"HOG visualisation  ({feat_len}-D descriptor)")
axes[1].axis('off')

plt.suptitle("HOG Feature Descriptor — INRIA Person sample", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4 · Dataset Loading & HOG Feature Extraction

The full INRIA Person dataset is loaded from HuggingFace (~239 MB, cached after the first run).  
Each PIL `image` is converted to a uint8 numpy array, resized to the detection window, and its **HOG descriptor** is computed.  
Labels are provided by the `imagefolder` loader: `0 = no_pedestrians`, `1 = pedestrians`.

In [ ]:
print(f"Loading '{CONFIG['dataset_id']}' from HuggingFace …")
print("(first run caches ~239 MB; subsequent runs are instant)\n")

ds = load_dataset(CONFIG['dataset_id'], split='train')

print(f"Total samples   : {len(ds)}")
print(f"Features        : {ds.features}")

# Resolve label names from the ClassLabel schema
try:
    label_names = ds.features['label'].names
except Exception:
    label_names = ['no_pedestrians', 'pedestrians']

print(f"\nLabel mapping   : {dict(enumerate(label_names))}")
print("\nClass distribution:")
counts = np.bincount([ex['label'] for ex in ds])
for lbl_id, (lbl_name, cnt) in enumerate(zip(label_names, counts)):
    print(f"  {lbl_id}  {lbl_name:<20}  {cnt} samples")


def pil_to_bgr(pil_img) -> np.ndarray:
    """Convert a PIL Image → uint8 BGR numpy array."""
    rgb = np.array(pil_img.convert('RGB'))
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)


def extract_hog_features(dataset) -> tuple:
    """Compute HOG descriptor for every sample in `dataset`.
    Returns (X, y) as float32 / int arrays."""
    X, y = [], []
    for ex in tqdm(dataset, desc='  Computing HOG'):
        img_bgr = pil_to_bgr(ex['image'])
        X.append(compute_hog(img_bgr))
        y.append(ex['label'])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


print("\nExtracting HOG features for all samples …")
X_all, y_all = extract_hog_features(ds)

print(f"\nFeature matrix : {X_all.shape}  (samples × dims)")
print(f"Label array    : {y_all.shape}  — unique={np.unique(y_all, return_counts=True)}")
print("\n✓ HOG extraction complete.")

---
## 5 · Build Training & Test Sets

Stratified split preserves the class ratio in both partitions.  
Index arrays (`train_idx` / `test_idx`) are kept so test patches can be linked back to the original PIL images for visualisation later.

In [ ]:
indices = np.arange(len(X_all))

train_idx, test_idx = train_test_split(
    indices,
    test_size=CONFIG['test_size'],
    random_state=CONFIG['random_seed'],
    stratify=y_all,
)

X_train, y_train = X_all[train_idx], y_all[train_idx]
X_test,  y_test  = X_all[test_idx],  y_all[test_idx]

print(f"Train : {len(X_train):5d} samples  "
      f"(pos={int(y_train.sum())}, neg={int((y_train==0).sum())})")
print(f"Test  : {len(X_test):5d} samples  "
      f"(pos={int(y_test.sum())}, neg={int((y_test==0).sum())})")
print(f"Dim   : {X_train.shape[1]}")

---
## 6 · SVM Training

A **StandardScaler** is applied first (zero mean, unit variance) — essential for RBF-kernel SVMs.

**Grid search** over `C` (regularisation) and `γ` (RBF width) is performed with 3-fold cross-validation to find the best hyper-parameters.

In [ ]:
# ── Feature scaling ────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ── Grid search over C and gamma ──────────────────────────────────────────
param_grid = {
    'C'     : [0.1, 1.0, 10.0],
    'gamma' : ['scale', 0.01, 0.001],
    'kernel': ['rbf'],
}

grid = GridSearchCV(
    SVC(probability=True, random_state=42),
    param_grid, cv=3, scoring='accuracy',
    n_jobs=-1, verbose=1,
)
grid.fit(X_train_s, y_train)

model = grid.best_estimator_
print(f"\nBest parameters : {grid.best_params_}")
print(f"Best CV accuracy: {grid.best_score_:.4f}")

# Save model and scaler
with open(CONFIG['model_path'],  'wb') as f: pickle.dump(model,  f)
with open(CONFIG['scaler_path'], 'wb') as f: pickle.dump(scaler, f)
print(f"\nModel saved → {CONFIG['model_path']}")
print(f"Scaler saved → {CONFIG['scaler_path']}")

---
## 7 · Evaluation

We compute:
- **Accuracy, Precision, Recall, F1** — per-class and macro averages.  
- **Confusion matrix** — TP / FP / FN / TN counts.  
- **ROC-AUC** — area under the receiver-operating-characteristic curve.

In [ ]:
y_pred       = model.predict(X_test_s)
y_pred_proba = model.predict_proba(X_test_s)[:, 1]

acc  = accuracy_score (y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score   (y_test, y_pred)
f1   = f1_score       (y_test, y_pred)
cm   = confusion_matrix(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

print("──────────────────────────────")
print(f"  Accuracy  : {acc :.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec :.4f}")
print(f"  F1-Score  : {f1  :.4f}")
print(f"  ROC-AUC   : {roc_auc:.4f}")
print("──────────────────────────────")
print()
print(classification_report(y_test, y_pred,
                             target_names=['Background', 'Pedestrian']))

---
## 8 · Results Visualisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# ── Confusion Matrix ───────────────────────────────────────────────────────
ax = axes[0, 0]
im = ax.imshow(cm, cmap='Blues')
fig.colorbar(im, ax=ax)
labels = ['Background', 'Pedestrian']
ax.set_xticks([0, 1]); ax.set_xticklabels(labels)
ax.set_yticks([0, 1]); ax.set_yticklabels(labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
thresh = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=14,
                color='white' if cm[i, j] > thresh else 'black')

# ── ROC Curve ─────────────────────────────────────────────────────────────
ax = axes[0, 1]
ax.plot(fpr, tpr, color='darkorange', lw=2,
        label=f'AUC = {roc_auc:.4f}')
ax.plot([0, 1], [0, 1], 'navy', lw=2, linestyle='--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

# ── Metric Bar Chart ──────────────────────────────────────────────────────
ax = axes[1, 0]
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values  = [acc, prec, rec, f1]
colors  = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c']
bars = ax.bar(metrics, values, color=colors, edgecolor='black')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Performance Metrics')
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, values):
    ax.annotate(f'{val:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points',
                ha='center', fontsize=11, fontweight='bold')

# ── Prediction Score Distribution ────────────────────────────────────────
ax = axes[1, 1]
ax.hist(y_pred_proba[y_test == 0], bins=30, alpha=0.7,
        label='Background', color='steelblue', density=True)
ax.hist(y_pred_proba[y_test == 1], bins=30, alpha=0.7,
        label='Pedestrian', color='tomato', density=True)
ax.axvline(0.5, color='green', linestyle='--', lw=2, label='Threshold = 0.5')
ax.set_xlabel('Predicted Probability for Pedestrian')
ax.set_ylabel('Density')
ax.set_title('Score Distribution')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('HOG + SVM Pedestrian Detector — Evaluation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('hog_svm_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Results saved → hog_svm_results.png")

---
## 9 · Prediction Visualisation on Test Patches

Display a random grid of 20 test patches with their **predicted** and **true** labels.  
- **Green border** — correct prediction  
- **Red border** — incorrect prediction (misclassification)

In [ ]:
np.random.seed(CONFIG['random_seed'])

# Run predictions on full test set
predictions = model.predict(X_test_s)

# Randomly sample 20 test indices for display
n_show    = 20
show_idx  = np.random.choice(len(X_test), n_show, replace=False)

fig, axes = plt.subplots(4, 5, figsize=(15, 13))

for plot_i, local_i in enumerate(show_idx):
    ax = axes[plot_i // 5, plot_i % 5]

    # Retrieve the original PIL image from the dataset
    ds_i      = int(test_idx[local_i])
    img_pil   = ds[ds_i]['image']
    img_rgb   = np.array(img_pil.convert('RGB'))

    pred  = int(predictions[local_i])
    truth = int(y_test[local_i])

    pred_name  = label_names[pred]  if pred  < len(label_names) else str(pred)
    truth_name = label_names[truth] if truth < len(label_names) else str(truth)

    ax.imshow(img_rgb)
    ax.set_title(
        f"Pred : {pred_name}\nTrue : {truth_name}",
        fontsize=7.5,
        color='#2ecc71' if pred == truth else '#e74c3c',
    )
    ax.axis('off')

    # Colour the border
    border_color = '#2ecc71' if pred == truth else '#e74c3c'
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(border_color)
        spine.set_linewidth(4)

n_correct = int((predictions == y_test).sum())
n_total   = len(y_test)

plt.suptitle(
    f"INRIA Person — Test Patch Predictions\n"
    f"Overall accuracy on {n_total} test samples: {n_correct}/{n_total} "
    f"({100*n_correct/n_total:.1f} %)\n"
    f"Green border = correct  ·  Red border = incorrect",
    fontsize=12, fontweight='bold',
)
plt.tight_layout()
plt.savefig('hog_svm_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → hog_svm_predictions.png")